In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Step 1: Imports

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


# Step 2: Load CIFAR-10

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

classes = train_dataset.classes
print(classes)

100%|██████████| 170M/170M [00:03<00:00, 42.9MB/s] 


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# Step 3: Vision Transformer Model

In [4]:
class SimpleViT(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=4,
        num_classes=10,
        embed_dim=128,
        depth=4,
        num_heads=4
    ):
        super().__init__()

        self.patch_size = patch_size

        num_patches = (image_size // patch_size) ** 2

        patch_dim = 3 * patch_size * patch_size

        self.patch_embedding = nn.Linear(
            patch_dim,
            embed_dim
        )

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                num_patches + 1,
                embed_dim
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.mlp_head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        B = x.shape[0]

        patches = x.unfold(
            2,
            self.patch_size,
            self.patch_size
        ).unfold(
            3,
            self.patch_size,
            self.patch_size
        )

        patches = patches.contiguous().view(
            B,
            3,
            -1,
            self.patch_size,
            self.patch_size
        )

        patches = patches.permute(
            0,
            2,
            1,
            3,
            4
        )

        patches = patches.reshape(
            B,
            -1,
            3 * self.patch_size * self.patch_size
        )

        x = self.patch_embedding(patches)

        cls_tokens = self.cls_token.expand(
            B,
            -1,
            -1
        )

        x = torch.cat(
            [cls_tokens, x],
            dim=1
        )

        x = x + self.pos_embedding

        x = self.transformer(x)

        cls_output = x[:, 0]

        return self.mlp_head(cls_output)

# Step 4: Create Model

In [5]:
model = SimpleViT().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

print(model)

SimpleViT(
  (patch_embedding): Linear(in_features=48, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (mlp_head): Linear(in_features=128, out_features=10, bias=True)
)


# Step 5: Training Loop

Start with only 20 epochs.

In [8]:
epochs = 20

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/20, Loss: 2.0398
Epoch 2/20, Loss: 2.0501
Epoch 3/20, Loss: 2.0335
Epoch 4/20, Loss: 2.0067
Epoch 5/20, Loss: 2.0477
Epoch 6/20, Loss: 1.9908
Epoch 7/20, Loss: 1.9749
Epoch 8/20, Loss: 1.9499
Epoch 9/20, Loss: 2.1080
Epoch 10/20, Loss: 2.1723
Epoch 11/20, Loss: 2.0479
Epoch 12/20, Loss: 1.9854
Epoch 13/20, Loss: 1.9574
Epoch 14/20, Loss: 1.9511
Epoch 15/20, Loss: 1.9214
Epoch 16/20, Loss: 1.9172
Epoch 17/20, Loss: 1.9126
Epoch 18/20, Loss: 1.9394
Epoch 19/20, Loss: 1.9326
Epoch 20/20, Loss: 1.9202


# Step 6: Test Accuracy

In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 27.67%


# Step 7: Predict One Image

In [10]:
image, label = test_dataset[0]

model.eval()

with torch.no_grad():

    prediction = model(
        image.unsqueeze(0).to(device)
    )

    predicted_class = prediction.argmax(
        dim=1
    ).item()

print("Actual:", classes[label])
print("Predicted:", classes[predicted_class])

Actual: cat
Predicted: frog


In [11]:
model.eval()

for i in range(10):

    image, label = test_dataset[i]

    with torch.no_grad():

        prediction = model(
            image.unsqueeze(0).to(device)
        )

        predicted_class = prediction.argmax(
            dim=1
        ).item()

    print(
        f"Actual: {classes[label]} | "
        f"Predicted: {classes[predicted_class]}"
    )

Actual: cat | Predicted: frog
Actual: ship | Predicted: ship
Actual: ship | Predicted: airplane
Actual: airplane | Predicted: ship
Actual: frog | Predicted: deer
Actual: frog | Predicted: deer
Actual: automobile | Predicted: automobile
Actual: frog | Predicted: deer
Actual: cat | Predicted: dog
Actual: automobile | Predicted: airplane
